### HMS GH200 On-Prem Model ###

In [14]:
import sys
import os
from dotenv import load_dotenv
import pandas as pd

# The new API version
from openai import OpenAI

%load_ext autoreload
%autoreload 2
import agentix
from agentix.hmsmodel import get_access_token, HMSModel

print(f'Package version: {agentix.__version__}')
print(f'Authors:         {agentix.__authors__}')
print(f'Python version:  {sys.version}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Package version: 0.0.1
Authors:         Andreas Werdich
The Core for Computational Biomedicine at Harvard Medical School
https://dbmi.hms.harvard.edu/about-dbmi/core-computational-biomedicine
Python version:  3.12.13 (main, Aug  5 2026, 01:10:24) [GCC 14.2.0]


In [2]:
# Load environmental variables
load_dotenv()
data_root = os.environ.get('HMS_API_ENDPOINT')
if data_root is None:
    print('Edit the "env"file and copy it to the root of the repository.') 
else:
    endpoint = os.environ['HMS_API_ENDPOINT']
    model = os.environ['HMS_MODEL_ID']
    hms_user = os.environ['HMS_USER']
    hms_pass = os.environ['HMS_PASS']
print(f'HMS endpoint: {endpoint}')
print(f'HMS model:    {model}')

HMS endpoint: https://ai-poc.hms.edu
HMS model:    muse-glimmer


#### HMS AI TOKEN ###

In [3]:
hms_ai_token = get_access_token(username=hms_user, password=hms_pass)
print(hms_ai_token)

eyJraWQiOiI3bmR6MXN3X1dFTjJrVC1Hdmx0ckpSZ2UxMEhlc2M3T2JuOGgyaVZQVk5VIiwiYWxnIjoiUlMyNTYifQ.eyJ2ZXIiOjEsImp0aSI6IkFULm9PMEg3NENKOHZOaExaTk93UHBWTWI1eHczUERWV056ajZQLU55NmlLWjQub2FyNGdrNDNyN0FKUVFpOWw2OTciLCJpc3MiOiJodHRwczovL2xvZ2luLmhtcy5oYXJ2YXJkLmVkdS9vYXV0aDIvYXVzMTU1bHp6cHR5RFRnTjM2OTgiLCJhdWQiOiJhcGk6Ly8wb2ExMzl0aXlsemJXNlhuWDY5OCIsImlhdCI6MTc4ODIwMTA0NiwiZXhwIjoxNzg4Mjg3NDQ2LCJjaWQiOiIwb2ExMzl0aXlsemJXNlhuWDY5OCIsInVpZCI6IjAwdTkxanVwdzRwNUd2UFY3Njk3Iiwic2NwIjpbIm9mZmxpbmVfYWNjZXNzIiwib3BlbmlkIl0sImF1dGhfdGltZSI6MTc4ODIwMTA0Niwic3ViIjoiQUFXMTBATUVELkhBUlZBUkQuRURVIiwiZ3JvdXBzIjpbIkRvbWFpbiBVc2VycyIsIkhQQ19MT05HV09PRC1VU0VSUyIsIkRZTl9RdWFkX01lbWJlcnNub1d5c3MiLCJEQk1JLUhTRE0tQVBQLUhTRE1FUkgtSU1HLTAxIiwiSFBDX0NDQl9TQUJBVElOSSIsIkhQQ19DQ0JfV0lOU1RPTiIsIkRyb3Bib3giLCJIUENfSElHSE1FTV9VU0VSUyIsIkhNUy1BenVyZS1PcGVuQUkgVXNlckFjY2VzcyIsIkhNUy1BenVyZS1PcGVuQUkgRGV2ZWxvcGVyIEFjY2VzcyIsIkhQQ19DQ0IiLCJFTVMgT1JJT04gRkFDVUxUWSIsIkhQQ19ET01BSU4tVVNFUlMiLCJsd191c2VycyIsIkhQQ19DQ0JfSEVNQkVSRyIsIkhQQ

### OpenAI ###
The v1 API simplifies authentication, removes the need for dated api-version parameters, and supports cross-provider model calls.

In [4]:
base_url = f'{endpoint}/v1'
client = OpenAI(base_url=base_url, api_key=hms_ai_token)

In [6]:
message = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Write a one-sentence poem about unicorns"}
]

response = client.chat.completions.create(model=model, messages=message)

print('Response')
print()
print(response.choices[0].message.content)
print()
print(f'finish_reason: {response.choices[0].finish_reason}')   # likely "length"
print()
usage = response.usage
print(usage.model_dump_json(indent=2))

Response

In the hush before dawn a unicorn steps through mist and leaves only the scent of rain where its hoof has been.

finish_reason: stop

{
  "completion_tokens": 289,
  "prompt_tokens": 43,
  "total_tokens": 332,
  "completion_tokens_details": null,
  "compute_units": null,
  "prompt_tokens_details": null
}


### HMSModel class ###

In [29]:
hms = HMSModel(token=hms_ai_token, model=model, base_url=base_url)
client = hms.create_client()
print(f'Current model: {hms.model}')

# Create the messages
user_prompt = 'You are a helpful AI assistant.'
system_prompt = 'Define the term artificial intelligence in a short paragraph'

message_list = hms.create_messages(user_prompt=user_prompt, system_prompt=system_prompt)
output = hms.chat_completion(messages=message_list)

print(message_list)
print()
print(output)

Current model: muse-glimmer
[{'role': 'system', 'content': 'Define the term artificial intelligence in a short paragraph'}, {'role': 'user', 'content': 'You are a helpful AI assistant.'}]

Artificial Intelligence, or AI, is the field of computer science focused on creating systems that can perform tasks normally requiring human intelligence. This includes learning from data, reasoning about problems, recognizing patterns, understanding language, and making decisions with limited guidance. AI systems are built using algorithms and models, such as machine learning and neural networks, that allow them to improve their performance over time based on experience rather than being explicitly programmed for every situation.
